# NB03 — normal-breast reference and reversal target

**In:** GTEx breast, TCGA adjacent normals, deconvolved intrinsic expression
**Out:** `data/interim/normal_reference.parquet`
**Gate:** diagnostic (no numeric threshold). Phase 1 checkpoint: Spearman of Q4 drug ranks bulk vs intrinsic.
**Runtime:** ~30 min


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe
try:
    import certifi, os
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
except Exception:
    pass

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = False
N_SAMPLES  = None   # full TCGA-BRCA; do not cap
N_SC_CELLS = 25_000  # Wu reference subsample if RAM is tight
N_PATIENTS = None
N_DRUGS    = None

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    if "sample_ids" not in kwargs:
        kwargs.setdefault("cohort", False)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config
NORMAL_OUT = INTERIM / "normal_reference.parquet"
Q4_DIR = REPO_ROOT / "results" / "mofa_clusters"
MOFA_CLUSTERS = REPO_ROOT / "outputs" / "mofa" / "mofa_clusters.csv"
import numpy as np, pandas as pd
from scipy.spatial.distance import cdist
from signatures import build_cluster_signature, rank_correlation_by_drug


In [ ]:
# Load
intr = INTERIM / "intrinsic_expression.parquet"
harm = INTERIM / "harmonised_expression.parquet"
expr = None
if intr.exists():
    expr = pd.read_parquet(intr)
elif harm.exists():
    expr = pd.read_parquet(harm)
    print("intrinsic missing; using harmonised bulk for the diagnostic")
gtex_files = [p for p in (RAW / "gtex_breast").glob("**/*") if p.is_file() and p.name != "PLACEHOLDER.txt"]
print("GTEx files", gtex_files)


In [ ]:
# Compute normal centroid (GTEx if present, else lowest-proliferation METABRIC tercile as a declared surrogate)
from io_data import pick_data_file, read_gct
gtex_expr = pick_data_file(RAW / "gtex_breast", "*.gct.gz", "*.gct", "*.parquet", "*.csv")
print("GTEx expression file", gtex_expr)
if expr is None:
    print("no expression matrix")
    normal = None
else:
    genes = expr.select_dtypes(include=[np.number]).columns
    X = expr[genes].apply(pd.to_numeric, errors="coerce")
    X.columns = X.columns.astype(str).str.upper()
    genes = X.columns
    if gtex_expr is not None:
        try:
            g = read_gct(gtex_expr) if "gct" in gtex_expr.name.lower() else (
                pd.read_parquet(gtex_expr) if gtex_expr.suffix == ".parquet" else pd.read_csv(gtex_expr, index_col=0)
            )
            g.columns = g.columns.astype(str).str.upper()
            common = X.columns.intersection(g.columns)
            normal_vec = g[common].mean(0)
            note_src = "GTEx"
        except Exception as e:
            print("GTEx parse failed", e)
            gtex_expr = None
    if gtex_expr is None:
        if "MKI67" in X.columns:
            normal_vec = X.loc[X["MKI67"].nsmallest(max(20, len(X)//10)).index, X.columns].mean(0)
            note_src = "low-MKI67 surrogate (GTEx missing)"
        else:
            normal_vec = X.mean(0)
            note_src = "cohort mean surrogate (GTEx missing)"
    normal = pd.DataFrame({"gene": genes, "normal_epithelial": normal_vec.reindex(genes).to_numpy(), "source": note_src})
    normal.to_parquet(NORMAL_OUT, index=False)
    print("normal source:", note_src)

    # cluster-centroid vs normal orthogonality
    if MOFA_CLUSTERS.exists():
        cl = pd.read_csv(MOFA_CLUSTERS)
        sid, col = cl.columns[0], "MOFA_CLUSTER"
        cl[sid] = cl[sid].astype(str)
        X.index = X.index.astype(str)
        common = X.index.intersection(cl[sid])
        cents = []
        labels = []
        for k, sub in cl[cl[sid].isin(common)].groupby(col):
            cents.append(X.loc[sub[sid].astype(str), genes].mean(0).to_numpy())
            labels.append(int(k))
        C = np.vstack(cents)
        nvec = normal_vec.reindex(genes).fillna(0).to_numpy()
        toward_normal = nvec - C
        # pairwise cluster directions
        angles = []
        for i in range(len(C)):
            others = C.mean(0) - C[i]
            a, b = toward_normal[i], others
            cos = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)
            angles.append(float(np.degrees(np.arccos(np.clip(cos, -1, 1)))))
        print("angle between 'toward normal' and 'away from other clusters' (deg):", dict(zip(labels, angles)))
        pd.Series(angles, index=labels, name="angle_deg").to_csv(INTERIM / "NB03_reversal_vs_normal_angles.csv")


In [ ]:
# Phase 1 checkpoint — Q4 rank correlation bulk vs intrinsic signatures
rho_mean = float("nan")
if expr is not None and MOFA_CLUSTERS.exists() and Q4_DIR.exists():
    cl = pd.read_csv(MOFA_CLUSTERS)
    sid = cl.columns[0]
    clusters = cl.set_index(sid)["MOFA_CLUSTER"]
    clusters.index = clusters.index.astype(str)
    # PAM50 from committed clinical if present
    clin_path = REPO_ROOT / "outputs" / "mofa" / "mofa_clinical_clusters.csv"
    pam50 = None
    if clin_path.exists():
        clin = pd.read_csv(clin_path)
        pam50 = clin.set_index("PATIENT_ID")["CLAUDIN_SUBTYPE"]
        pam50.index = pam50.index.astype(str)
    X = expr.select_dtypes(include=[np.number])
    X.index = X.index.astype(str)
    rhos = []
    for k in sorted(clusters.unique()):
        q4 = Q4_DIR / f"cluster_{int(k)}_drug_targets.csv"
        if not q4.exists() or pam50 is None:
            continue
        sig = build_cluster_signature(X, clusters, pam50, int(k))
        # compare gene-level coef ranks vs committed signature as a proxy when GCTX is absent
        old_sig = Q4_DIR / f"cluster_{int(k)}_signature.csv"
        if old_sig.exists():
            old = pd.read_csv(old_sig).set_index("gene")["coef"]
            new = sig.set_index("gene")["coef"]
            rho = rank_correlation_by_drug(old, new)
            rhos.append(rho)
            print(f"cluster {k} signature Spearman bulk-committed vs intrinsic: {rho:.3f}")
        drugs = pd.read_csv(q4)
        drugs.to_csv(INTERIM / f"NB03_cluster_{int(k)}_v1_ranks.csv", index=False)
    if rhos:
        rho_mean = float(np.nanmean(rhos))
        print("mean signature Spearman", rho_mean)
pd.Series({"mean_signature_spearman": rho_mean}).to_json(INTERIM / "NB03_phase1_checkpoint.json")


In [ ]:
# GATE — diagnostic notebook: log the checkpoint, no hard fail threshold
val = 0.0 if pd.isna(rho_mean) else float(rho_mean)
gate("NB03", "phase1_rank_delta_logged", 1.0, 1.0,
     n=None if pd.isna(rho_mean) else 1,
     note=f"mean signature Spearman vs committed Q4 signatures={val:.4f} (no threshold; both 'moved a lot' and 'barely moved' are publishable)")


In [ ]:
# Figures
try:
    import matplotlib.pyplot as plt
    ang = INTERIM / "NB03_reversal_vs_normal_angles.csv"
    if ang.exists():
        s = pd.read_csv(ang, index_col=0).squeeze()
        fig, ax = plt.subplots(figsize=(4, 3))
        ax.bar([str(i) for i in s.index], s.values)
        ax.set_ylabel("angle (deg)"); ax.set_xlabel("MOFA cluster")
        ax.set_title("Toward-normal vs away-from-other-clusters")
        ax.axhline(90, ls="--", c="gray")
        fig.tight_layout(); fig.savefig(FIGURES / "NB03_normal_vs_cluster.png", dpi=140)
except Exception as e:
    print(e)
